# Practical Session 03
## Part I - Instructor-led mini-lab
### Example 1 - Data arrays, scaling, and a baseline

We start from a small supervised regression problem. The goal is not to use a sophisticated model, but to keep the numerical structure visible.

Before running the code, think about the following questions:

1. Which axis labels samples?
2. Which axis labels features?
3. Which quantities are allowed to be computed from the training set only?
4. What is a useful baseline before fitting any model?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
rng = np.random.default_rng(2016)

N = 600

time = rng.uniform(0.0, 10.0, size=N)
field = rng.uniform(-1.0, 1.0, size=N)
temperature = 300.0 + 5.0 * rng.normal(size=N)

X = np.column_stack([time, field, temperature])

theta_true = np.array([0.45, -1.20, 0.08])
bias_true = -23.0
noise = 0.25 * rng.normal(size=N)

y = X @ theta_true + bias_true + noise

print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("feature means:", X.mean(axis=0))
print("feature scales:", X.std(axis=0))

assert X.ndim == 2
assert y.ndim == 1
assert X.shape[0] == y.shape[0]


The features have different numerical scales. The optimizer will see these numbers directly, so preprocessing is part of the numerical experiment.

We now split the data and fit the standardization only on the training subset.


In [ ]:
idx = rng.permutation(N)

n_train = int(0.70 * N)
n_val = int(0.15 * N)

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train + n_val]
test_idx = idx[n_train + n_val:]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

mu = X_train.mean(axis=0)
scale = X_train.std(axis=0)
scale[scale == 0.0] = 1.0

X_train_s = (X_train - mu) / scale
X_val_s = (X_val - mu) / scale
X_test_s = (X_test - mu) / scale

print("X_train_s.shape:", X_train_s.shape)
print("X_val_s.shape:  ", X_val_s.shape)
print("X_test_s.shape: ", X_test_s.shape)
print("training mean after scaling:", X_train_s.mean(axis=0))
print("training std after scaling: ", X_train_s.std(axis=0))

assert X_train_s.shape == X_train.shape
assert X_val_s.shape == X_val.shape
assert X_test_s.shape == X_test.shape


In [ ]:
def mse_loss(y_pred, y):
    residual = y_pred - y
    return 0.5 * np.mean(residual**2)

baseline = y_train.mean()

train_baseline_loss = mse_loss(np.full_like(y_train, baseline), y_train)
val_baseline_loss = mse_loss(np.full_like(y_val, baseline), y_val)

print(f"training baseline loss:   {train_baseline_loss:.6f}")
print(f"validation baseline loss: {val_baseline_loss:.6f}")

Questions:

1. Why is the validation set not used when computing `mu` and `scale`?
2. Why is the mean predictor a useful baseline?
3. Which objects would you save if you wanted to reproduce this numerical experiment later?


### Example 2 - A minimal gradient-descent training loop

For a linear model

$$
\hat{y}=X\theta+b,
$$

the residual and mean squared loss are

$$
r=X\theta+b-y,
\qquad
L(\theta,b)=\frac{1}{2N}\sum_{i=1}^N r_i^2.
$$

The analytical gradients are

$$
\nabla_\theta L=\frac{1}{N}X^T r,
\qquad
\frac{\partial L}{\partial b}=\frac{1}{N}\sum_{i=1}^N r_i.
$$


In [ ]:
def linear_predict(X, theta, bias):
    return X @ theta + bias


def linear_gradients(X, y, theta, bias):
    y_pred = linear_predict(X, theta, bias)
    residual = y_pred - y

    grad_theta = X.T @ residual / residual.size
    grad_bias = residual.mean()

    return grad_theta, grad_bias


def train_linear_regression(X_train, y_train, X_val, y_val, eta=0.05, n_steps=500):
    theta = np.zeros(X_train.shape[1])
    bias = 0.0

    train_history = []
    val_history = []
    grad_norm_history = []

    for step in range(n_steps):
        y_train_pred = linear_predict(X_train, theta, bias)
        train_loss = mse_loss(y_train_pred, y_train)

        grad_theta, grad_bias = linear_gradients(X_train, y_train, theta, bias)
        grad_norm = np.sqrt(np.sum(grad_theta**2) + grad_bias**2)

        theta -= eta * grad_theta
        bias -= eta * grad_bias

        y_val_pred = linear_predict(X_val, theta, bias)
        val_loss = mse_loss(y_val_pred, y_val)

        train_history.append(train_loss)
        val_history.append(val_loss)
        grad_norm_history.append(grad_norm)

        if not np.isfinite(train_loss) or not np.isfinite(val_loss):
            break

    return {
        "theta": theta,
        "bias": bias,
        "train_history": np.array(train_history),
        "val_history": np.array(val_history),
        "grad_norm_history": np.array(grad_norm_history),
    }

In [ ]:
result = train_linear_regression(
    X_train_s,
    y_train,
    X_val_s,
    y_val,
    eta=0.05,
    n_steps=600,
)

print("theta in scaled coordinates:", result["theta"])
print("bias:", result["bias"])
print(f"final training loss:   {result['train_history'][-1]:.6f}")
print(f"final validation loss: {result['val_history'][-1]:.6f}")
print(f"final gradient norm:   {result['grad_norm_history'][-1]:.3e}")
print("finite training history:", np.isfinite(result["train_history"]).all())

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.3))

ax.plot(result["train_history"], label="training")
ax.plot(result["val_history"], label="validation")

ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_yscale("log")
ax.set_title("Gradient descent diagnostics")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()

Questions:

1. Which part of the code is the forward computation?
2. Which arrays have the same shape as the model parameters?
3. Why do we monitor validation loss even though it is not used in the update?


### Example 3 - Gradient checks and numerical scale

Finite differences are too expensive for training large models, but they are excellent sanity checks for small gradient calculations.

For one parameter vector, compare the analytical gradient with a central finite-difference approximation.


In [ ]:
def finite_difference_grad(loss_fn, theta, eps=1.0e-6):
    grad = np.zeros_like(theta)

    for j in range(theta.size):
        step = np.zeros_like(theta)
        step[j] = eps
        grad[j] = (loss_fn(theta + step) - loss_fn(theta - step)) / (2.0 * eps)

    return grad


theta0 = np.array([0.1, -0.2, 0.3])
bias0 = 0.05

loss_theta_only = lambda th: mse_loss(linear_predict(X_train_s, th, bias0), y_train)

grad_fd = finite_difference_grad(loss_theta_only, theta0)
grad_an, grad_b = linear_gradients(X_train_s, y_train, theta0, bias0)

rel_error = np.linalg.norm(grad_fd - grad_an) / np.linalg.norm(grad_fd + grad_an)

print("finite-difference gradient:", grad_fd)
print("analytical gradient:       ", grad_an)
print(f"relative difference: {rel_error:.3e}")

Now use the same learning rate on a well-scaled and an artificially badly-scaled representation of the same data. The mathematical information is similar, but the numerical optimization problem is not.


In [ ]:
def train_for_scale_demo(X_train, y_train, X_val, y_val, eta, n_steps=80):
    theta = np.zeros(X_train.shape[1])
    bias = 0.0
    history = []

    for step in range(n_steps):
        y_pred = linear_predict(X_train, theta, bias)
        loss = mse_loss(y_pred, y_train)
        history.append(loss)

        if (not np.isfinite(loss)) or loss > 1.0e80:
            break

        grad_theta, grad_bias = linear_gradients(X_train, y_train, theta, bias)
        theta -= eta * grad_theta
        bias -= eta * grad_bias

    return np.array(history)


X_train_bad = X_train_s.copy()
X_val_bad = X_val_s.copy()
X_train_bad[:, 0] *= 1.0e3
X_val_bad[:, 0] *= 1.0e3

eta = 0.05
hist_scaled = train_for_scale_demo(X_train_s, y_train, X_val_s, y_val, eta=eta)
hist_bad = train_for_scale_demo(X_train_bad, y_train, X_val_bad, y_val, eta=eta)

print("scaled representation:")
print("  steps recorded:", hist_scaled.size)
print("  last loss:     ", hist_scaled[-1])
print("  all finite:    ", np.isfinite(hist_scaled).all())

print("badly scaled representation:")
print("  steps recorded:", hist_bad.size)
print("  last loss:     ", hist_bad[-1])
print("  all finite:    ", np.isfinite(hist_bad).all())

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.3))

ax.plot(hist_scaled, label="scaled features")
ax.plot(hist_bad, label="one feature multiplied by 1000")

ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_yscale("log")
ax.set_title(fr"Same learning rate, $\eta={eta}$")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()

Questions:

1. Why can the same learning rate be reasonable in one representation and unstable in another?
2. Which diagnostic first reveals the unstable run?
3. How is this similar to choosing a time step in a numerical ODE solver?


### Example 4 - Equation residuals as scientific diagnostics

A small data loss is not the only possible criterion. In scientific machine learning, we may also ask whether a model respects a known equation.

For the function

$$
u(x)=\sin(\pi x),
$$

the equation

$$
u_{xx}+\pi^2u=0
$$

is satisfied exactly. A visually small perturbation can produce a much larger equation residual.


In [ ]:
x_eq = np.linspace(0.0, 1.0, 500)

good = np.sin(np.pi * x_eq)
perturbed = good + 0.03 * np.sin(8.0 * np.pi * x_eq)


def helmholtz_residual_1d(x, u):
    dx = x[1] - x[0]
    uxx = (u[2:] - 2.0 * u[1:-1] + u[:-2]) / dx**2
    residual = uxx + np.pi**2 * u[1:-1]
    return x[1:-1], residual


xi, R_good = helmholtz_residual_1d(x_eq, good)
_, R_perturbed = helmholtz_residual_1d(x_eq, perturbed)

field_mismatch = np.sqrt(np.mean((perturbed - good)**2))
res_good = np.sqrt(np.mean(R_good**2))
res_perturbed = np.sqrt(np.mean(R_perturbed**2))

print(f"RMS field mismatch:       {field_mismatch:.3e}")
print(f"RMS equation residual, good field:      {res_good:.3e}")
print(f"RMS equation residual, perturbed field: {res_perturbed:.3e}")

In [ ]:
fig, axes = plt.subplots(2, 1, sharex=False, figsize=(6.0, 8.4))

axes[0].plot(x_eq, good, label=r"$\sin(\pi x)$")
axes[0].plot(x_eq, perturbed, "--", label="small perturbation")
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$u(x)$")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].axhline(0.0, linewidth=1)
axes[1].plot(xi, R_good, label="residual: good")
axes[1].plot(xi, R_perturbed, label="residual: perturbed")
axes[1].set_xlabel(r"$x$")
axes[1].set_ylabel(r"$u_{xx}+\pi^2u$")
axes[1].set_yscale("symlog", linthresh=1.0e-6)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()

Questions:

1. Why can two curves look close while one violates the equation much more strongly?
2. Which diagnostic would be hidden if we only plotted the model output?
3. How could an equation residual be added to a training loss?


## Part II - Independent work


### Task 1 - Honest preprocessing, baseline, and conditioning

Use the following data-generation cell:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)

N = 800

length = rng.uniform(0.0, 2.0, size=N)
energy = rng.uniform(1.0e3, 5.0e3, size=N)
angle = rng.uniform(-1.0, 1.0, size=N)

X = np.column_stack([length, energy, angle])

theta_true = np.array([2.0, -1.5e-3, 0.8])
bias_true = 0.4
noise = 0.12 * rng.normal(size=N)

y = X @ theta_true + bias_true + noise

Your tasks are:

1. State the meaning of axis 0 and axis 1 of `X`.
2. Split the data into training, validation, and test sets using a fixed random permutation. Use approximately 70%, 15%, and 15% of the samples.
3. Compute the feature mean and standard deviation from the training set only.
4. Use these quantities to standardize the training, validation, and test sets.
5. Add array contracts checking shapes, finite values, and that the standardized training features have approximately zero mean and unit standard deviation.
6. Compute the training and validation loss of a mean-prediction baseline.
7. Fit a linear model using `np.linalg.lstsq` on the standardized training data. Include the bias by adding a column of ones.
8. Compute training, validation, and test losses for the fitted model.
9. Compute the condition number of

   $$
   H=\frac{1}{N}X^TX
   $$

before and after standardization. Use the training data.  
10. In two or three sentences, explain why scaling is not only cosmetic in this problem.

**Optional extension**: Repeat the standardization incorrectly by using all data before the split. Explain why this creates data leakage even if the final numbers change only slightly.


In [ ]:
#####################
# Your solution here
#####################

### Task 2 - Gradient check and learning-rate diagnosis

Use the following setup:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(77)

N = 350

x1 = rng.normal(size=N)
x2 = 0.5 * rng.normal(size=N)
X = np.column_stack([x1, x2])

theta_true = np.array([1.5, -2.0])
bias_true = -0.4

y = X @ theta_true + bias_true + 0.15 * rng.normal(size=N)

# The features are already on a reasonable scale for this task.
print("X.shape:", X.shape)
print("y.shape:", y.shape)

Your tasks are:

1. Write a function `predict(X, theta, bias)`.
2. Write a function `loss(X, y, theta, bias)` using the factor `0.5` in front of the mean squared residual.
3. Write a function `gradients(X, y, theta, bias)` returning `grad_theta` and `grad_bias`.
4. Check that `grad_theta.shape == theta.shape`.
5. Implement a central finite-difference gradient check for `theta` at a nonzero test point, for example:

   ```python
   theta0 = np.array([0.2, -0.3])
   bias0 = 0.1
   ```

6. Report the relative difference between the finite-difference gradient and the analytical gradient.
7. Implement gradient descent and run it for several learning rates, for example:

   ```python
   eta_values = [0.001, 0.03, 0.3, 1.0]
   ```

8. Plot the loss histories on a logarithmic scale.
9. Identify which learning rates are stable, slow, oscillatory, or divergent.
10. Add explicit checks for non-finite losses and gradients.

In [ ]:
#####################
# Your solution here
#####################

### Task 3 - A small physics-informed learning workflow

We observe noisy samples of a function on the interval $[0,1]$. The underlying clean function approximately satisfies the Helmholtz equation

$$
u_{xx}+\pi^2u=0.
$$

We will fit the observations with a sine basis

$$
u_\theta(x)=\sum_{m=1}^{M}\theta_m\sin(m\pi x).
$$

The model is linear in the parameters, but the workflow has the same structure as a small machine-learning experiment: data, model, loss, gradient, validation, and residual diagnostics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2027)

n_data = 90
x_data = np.sort(rng.uniform(0.0, 1.0, size=n_data))

u_clean = np.sin(np.pi * x_data)
sigma = 0.04 * (1.0 + 0.5 * x_data)
u_obs = u_clean + sigma * rng.normal(size=n_data)

idx = rng.permutation(n_data)
n_train = int(0.65 * n_data)
n_val = int(0.20 * n_data)

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train + n_val]
test_idx = idx[n_train + n_val:]

modes = np.arange(1, 9)


def basis(x, modes):
    return np.sin(np.pi * x[:, None] * modes[None, :])


def residual_basis(x, modes):
    # Basis for the normalized residual (u_xx + pi^2 u) / pi^2.
    Phi = basis(x, modes)
    factors = 1.0 - modes**2
    return Phi * factors[None, :]


x_collocation = np.linspace(0.0, 1.0, 300)
x_plot = np.linspace(0.0, 1.0, 500)
u_true_plot = np.sin(np.pi * x_plot)

print("number of modes:", modes.size)
print("training samples:", train_idx.size)
print("validation samples:", val_idx.size)
print("test samples:", test_idx.size)

Your tasks are:

1. Construct the basis matrices for the training, validation, test, collocation, and plotting points.
2. Write a function `predict(Phi, theta)` returning `Phi @ theta`.
3. Define a weighted data loss on a subset:

   $$
   L_{\mathrm{data}}=\frac{1}{2N}\sum_i\left(\frac{u_\theta(x_i)-u_i}{\sigma_i}\right)^2.
   $$

4. Define a normalized equation-residual loss on the collocation grid:

   $$
   L_{\mathrm{pde}}=\frac{1}{2N_c}\sum_j\left(\frac{u_{\theta,xx}(x_j)+\pi^2u_\theta(x_j)}{\pi^2}\right)^2.
   $$

   The helper function `residual_basis` already gives the basis for the normalized residual.
5. Define the combined loss

   $$
   L = L_{\mathrm{data}} + \alpha L_{\mathrm{pde}} + \frac{\lambda}{2}\|\theta\|^2.
   $$

6. Derive and implement the gradient of this combined loss with respect to `theta`.
7. Train the model with gradient descent for at least three values of `alpha`, for example:

   ```python
   alpha_values = [0.0, 0.1, 1.0]
   lambda_reg = 1.0e-5
   eta = 5.0e-4
   n_steps = 3000
   ```

   If a run becomes non-finite, reduce the learning rate and record that this was necessary.
8. During training, record:
   - total training loss;
   - training data loss;
   - validation data loss;
   - PDE residual loss;
   - gradient norm.
9. Plot the training and validation data-loss histories for each value of `alpha`.
10. Plot the noisy data with error bars and overlay the fitted functions on a dense grid.
11. For each value of `alpha`, compute and report:
    - final training data loss;
    - validation data loss;
    - test data loss;
    - PDE residual loss;
    - RMS test error in physical units, not divided by `sigma`.
12. Plot the equation residual as a function of `x` for the fitted models.
13. Inspect the learned coefficients `theta`. Which modes are suppressed when `alpha` is increased?
14. Write a short scientific interpretation. Does the physics-informed loss improve the model, over-constrain it, or mainly suppress noisy high-frequency components?

In [ ]:
#####################
# Your solution here
#####################